## 1. import

In [9]:
import numpy as np
import pandas as pd

## 2. config

In [10]:
class CFG:
    SEED = 42
    N_SPLITS = 5
    TARGET_COL = "Irrigation_Need"
    ID_COL = "id"

    TRAIN_PATH = "/kaggle/input/competitions/playground-series-s6e4/train.csv"
    TEST_PATH = "/kaggle/input/competitions/playground-series-s6e4/test.csv"
    NPY_PATH = "/kaggle/input/datasets/mizushimatoshihiko/npy-files/"

## 3. models dict

In [11]:
# ==============================
# Load OOF / PRED
# ==============================
model_dict = {
    "cat_strict_baseline_bias": (
        np.load(CFG.NPY_PATH + "oof_catboost_strict_baseline_bias_proba.npy"),
        np.load(CFG.NPY_PATH + "pred_catboost_strict_baseline_bias_proba.npy"),
    ),
    "cat_strict_baseline_bias_wide": (
        np.load(CFG.NPY_PATH + "oof_catboost_strict_baseline_bias_wide_proba.npy"),
        np.load(CFG.NPY_PATH + "pred_catboost_strict_baseline_bias_wide_proba.npy"),
    ),
    "cat_digit_min_bias": (
        np.load(CFG.NPY_PATH + "oof_catboost_digit_min_bias_proba.npy"),
        np.load(CFG.NPY_PATH + "pred_catboost_digit_min_bias_proba.npy"),
    ),
    "cat_log_min_bias": (
        np.load(CFG.NPY_PATH + "oof_catboost_log_min_bias_proba.npy"),
        np.load(CFG.NPY_PATH + "pred_catboost_log_min_bias_proba.npy"),
    ),
    "cat_pairwise_te_bias_from_shared_min_proba": (
        np.load(CFG.NPY_PATH + "oof_cat_pairwise_te_bias_from_shared_min_proba_biased.npy"),
        np.load(CFG.NPY_PATH + "pred_cat_pairwise_te_bias_from_shared_min_proba_biased.npy"),
    ),
    "xgb_digit_orderedte_min_biased": (
        np.load(CFG.NPY_PATH + "oof_xgb_digit_orderedte_min_proba_biased.npy"),
        np.load(CFG.NPY_PATH + "pred_xgb_digit_orderedte_min_proba_biased.npy"),
    ),
    "lgb_digit_te_min_biased": (
        np.load(CFG.NPY_PATH + "oof_lgb_digit_te_min_proba_biased.npy"),
        np.load(CFG.NPY_PATH + "pred_lgb_digit_te_min_proba_biased.npy"),
    ),
}

target_name = "cat_pairwise_te_bias_from_shared_min_proba"

oof_t, pred_t = model_dict[target_name]

# shared の列順を base 側に合わせる
perm = [2, 0, 1]

model_dict[target_name] = (
    oof_t[:, perm],
    pred_t[:, perm],
)

target_name = "xgb_digit_orderedte_min_biased"

oof_t, pred_t = model_dict[target_name]

# shared の列順を base 側に合わせる
perm = [2, 0, 1]

model_dict[target_name] = (
    oof_t[:, perm],
    pred_t[:, perm],
)

target_name = "lgb_digit_te_min_biased"

oof_t, pred_t = model_dict[target_name]

# shared の列順を base 側に合わせる
perm = [2, 0, 1]

model_dict[target_name] = (
    oof_t[:, perm],
    pred_t[:, perm],
)

## 4. show correlations

In [12]:
# ==============================
# Helpers
# ==============================
def flatten_multiclass_proba(arr: np.ndarray) -> np.ndarray:
    """
    (n_samples, n_classes) -> 1d flatten
    相関確認用。まずは全クラスまとめて見る。
    """
    if arr.ndim == 1:
        return arr.astype(np.float64)
    return arr.reshape(-1).astype(np.float64)

def rank01_pd(x: np.ndarray) -> np.ndarray:
    s = pd.Series(x)
    r = s.rank(method="average").values.astype(np.float64)
    denom = max(len(r) - 1, 1)
    return (r - 1.0) / denom

# ==============================
# Build DataFrames
# ==============================
names = list(model_dict.keys())

oof_flat = [flatten_multiclass_proba(model_dict[k][0]) for k in names]
pred_flat = [flatten_multiclass_proba(model_dict[k][1]) for k in names]

oof_lens = [len(x) for x in oof_flat]
pred_lens = [len(x) for x in pred_flat]

if len(set(oof_lens)) != 1:
    raise ValueError(f"OOF lengths mismatch: {dict(zip(names, oof_lens))}")
if len(set(pred_lens)) != 1:
    raise ValueError(f"PRED lengths mismatch: {dict(zip(names, pred_lens))}")

df_oof = pd.DataFrame(np.stack(oof_flat, axis=1), columns=names)
df_pred = pd.DataFrame(np.stack(pred_flat, axis=1), columns=names)

print("df_oof shape :", df_oof.shape)
print("df_pred shape:", df_pred.shape)

# ==============================
# Raw correlation
# ==============================
print("\n=== OOF Pearson corr (raw) ===")
display(df_oof.corr())

print("\n=== TEST Pearson corr (raw) ===")
display(df_pred.corr())

# ==============================
# Rank correlation
# ==============================
df_oof_rank = df_oof.apply(lambda col: rank01_pd(col.values), axis=0, result_type="expand")
df_oof_rank.columns = names

df_pred_rank = df_pred.apply(lambda col: rank01_pd(col.values), axis=0, result_type="expand")
df_pred_rank.columns = names

print("\n=== OOF corr (rank -> Pearson) ===")
display(df_oof_rank.corr())

print("\n=== TEST corr (rank -> Pearson) ===")
display(df_pred_rank.corr())

# ==============================
# Focused view
# ==============================
target = "lgb_digit_te_min_biased"
base_models = [model for model in model_dict.keys() if model != target]

print(f"\n=== Focused correlations vs {target} ===")
for bm in base_models:
    print(
        f"{bm:30s} | "
        f"OOF raw={df_oof[target].corr(df_oof[bm]):.6f} | "
        f"TEST raw={df_pred[target].corr(df_pred[bm]):.6f} | "
        f"OOF rank={df_oof_rank[target].corr(df_oof_rank[bm]):.6f} | "
        f"TEST rank={df_pred_rank[target].corr(df_pred_rank[bm]):.6f}"
    )

df_oof shape : (1890000, 7)
df_pred shape: (810000, 7)

=== OOF Pearson corr (raw) ===


,cat_strict_baseline_bias,cat_strict_baseline_bias_wide,cat_digit_min_bias,cat_log_min_bias,cat_pairwise_te_bias_from_shared_min_proba,xgb_digit_orderedte_min_biased,lgb_digit_te_min_biased
cat_strict_baseline_bias,1.000000,0.998880,0.998390,0.998878,0.989241,0.992234,0.990408
cat_strict_baseline_bias_wide,0.998880,1.000000,0.998953,0.999956,0.988350,0.991290,0.989505
cat_digit_min_bias,0.998390,0.998953,1.000000,0.998955,0.989264,0.992201,0.990410
cat_log_min_bias,0.998878,0.999956,0.998955,1.000000,0.988338,0.991278,0.989495
cat_pairwise_te_bias_from_shared_min_proba,0.989241,0.988350,0.989264,0.988338,1.000000,0.995078,0.994775
xgb_digit_orderedte_min_biased,0.992234,0.991290,0.992201,0.991278,0.995078,1.000000,0.998641
lgb_digit_te_min_biased,0.990408,0.989505,0.990410,0.989495,0.994775,0.998641,1.000000



=== TEST Pearson corr (raw) ===


,cat_strict_baseline_bias,cat_strict_baseline_bias_wide,cat_digit_min_bias,cat_log_min_bias,cat_pairwise_te_bias_from_shared_min_proba,xgb_digit_orderedte_min_biased,lgb_digit_te_min_biased
cat_strict_baseline_bias,1.000000,0.999206,0.998818,0.999202,0.989962,0.992737,0.991070
cat_strict_baseline_bias_wide,0.999206,1.000000,0.999369,0.999991,0.989230,0.991989,0.990338
cat_digit_min_bias,0.998818,0.999369,1.000000,0.999371,0.989904,0.992639,0.990983
cat_log_min_bias,0.999202,0.999991,0.999371,1.000000,0.989234,0.991989,0.990340
cat_pairwise_te_bias_from_shared_min_proba,0.989962,0.989230,0.989904,0.989234,1.000000,0.996178,0.995935
xgb_digit_orderedte_min_biased,0.992737,0.991989,0.992639,0.991989,0.996178,1.000000,0.999317
lgb_digit_te_min_biased,0.991070,0.990338,0.990983,0.990340,0.995935,0.999317,1.000000



=== OOF corr (rank -> Pearson) ===


,cat_strict_baseline_bias,cat_strict_baseline_bias_wide,cat_digit_min_bias,cat_log_min_bias,cat_pairwise_te_bias_from_shared_min_proba,xgb_digit_orderedte_min_biased,lgb_digit_te_min_biased
cat_strict_baseline_bias,1.000000,0.996560,0.994903,0.996545,0.906433,0.969500,0.960105
cat_strict_baseline_bias_wide,0.996560,1.000000,0.995681,0.999349,0.905451,0.968352,0.959154
cat_digit_min_bias,0.994903,0.995681,1.000000,0.995625,0.907748,0.969265,0.960066
cat_log_min_bias,0.996545,0.999349,0.995625,1.000000,0.905285,0.968147,0.958881
cat_pairwise_te_bias_from_shared_min_proba,0.906433,0.905451,0.907748,0.905285,1.000000,0.942156,0.945456
xgb_digit_orderedte_min_biased,0.969500,0.968352,0.969265,0.968147,0.942156,1.000000,0.992057
lgb_digit_te_min_biased,0.960105,0.959154,0.960066,0.958881,0.945456,0.992057,1.000000



=== TEST corr (rank -> Pearson) ===


,cat_strict_baseline_bias,cat_strict_baseline_bias_wide,cat_digit_min_bias,cat_log_min_bias,cat_pairwise_te_bias_from_shared_min_proba,xgb_digit_orderedte_min_biased,lgb_digit_te_min_biased
cat_strict_baseline_bias,1.000000,0.998150,0.996698,0.998165,0.908160,0.971433,0.963442
cat_strict_baseline_bias_wide,0.998150,1.000000,0.997234,0.999861,0.907338,0.970411,0.962579
cat_digit_min_bias,0.996698,0.997234,1.000000,0.997217,0.909781,0.971729,0.963901
cat_log_min_bias,0.998165,0.999861,0.997217,1.000000,0.907182,0.970298,0.962395
cat_pairwise_te_bias_from_shared_min_proba,0.908160,0.907338,0.909781,0.907182,1.000000,0.945409,0.952764
xgb_digit_orderedte_min_biased,0.971433,0.970411,0.971729,0.970298,0.945409,1.000000,0.996804
lgb_digit_te_min_biased,0.963442,0.962579,0.963901,0.962395,0.952764,0.996804,1.000000



=== Focused correlations vs lgb_digit_te_min_biased ===
cat_strict_baseline_bias       | OOF raw=0.990408 | TEST raw=0.991070 | OOF rank=0.960105 | TEST rank=0.963442
cat_strict_baseline_bias_wide  | OOF raw=0.989505 | TEST raw=0.990338 | OOF rank=0.959154 | TEST rank=0.962579
cat_digit_min_bias             | OOF raw=0.990410 | TEST raw=0.990983 | OOF rank=0.960066 | TEST rank=0.963901
cat_log_min_bias               | OOF raw=0.989495 | TEST raw=0.990340 | OOF rank=0.958881 | TEST rank=0.962395
cat_pairwise_te_bias_from_shared_min_proba | OOF raw=0.994775 | TEST raw=0.995935 | OOF rank=0.945456 | TEST rank=0.952764
xgb_digit_orderedte_min_biased | OOF raw=0.998641 | TEST raw=0.999317 | OOF rank=0.992057 | TEST rank=0.996804


In [13]:
# 3x3 の classwise cross-corr を見る
for bm in base_models:
    print(f"\n=== {bm} vs {target} : OOF classwise cross-corr ===")
    a = model_dict[bm][0]   # base model OOF
    b = model_dict[target][0]  # target model OOF

    mat = np.zeros((a.shape[1], b.shape[1]))
    for i in range(a.shape[1]):
        for j in range(b.shape[1]):
            mat[i, j] = np.corrcoef(a[:, i], b[:, j])[0, 1]

    display(pd.DataFrame(
        mat,
        index=[f"{bm}_class{i}" for i in range(a.shape[1])],
        columns=[f"{target}_class{j}" for j in range(b.shape[1])]
    ))


=== cat_strict_baseline_bias vs lgb_digit_te_min_biased : OOF classwise cross-corr ===


,lgb_digit_te_min_biased_class0,lgb_digit_te_min_biased_class1,lgb_digit_te_min_biased_class2
cat_strict_baseline_bias_class0,0.938621,-0.237892,-0.127677
cat_strict_baseline_bias_class1,-0.291167,0.995848,-0.919504
cat_strict_baseline_bias_class2,-0.041632,-0.932227,0.986332



=== cat_strict_baseline_bias_wide vs lgb_digit_te_min_biased : OOF classwise cross-corr ===


,lgb_digit_te_min_biased_class0,lgb_digit_te_min_biased_class1,lgb_digit_te_min_biased_class2
cat_strict_baseline_bias_wide_class0,0.935471,-0.238520,-0.125764
cat_strict_baseline_bias_wide_class1,-0.291219,0.995038,-0.918640
cat_strict_baseline_bias_wide_class2,-0.039561,-0.931770,0.985029



=== cat_digit_min_bias vs lgb_digit_te_min_biased : OOF classwise cross-corr ===


...

,lgb_digit_te_min_biased_class0,lgb_digit_te_min_biased_class1,lgb_digit_te_min_biased_class2
cat_log_min_bias_class0,0.935379,-0.238518,-0.125730
cat_log_min_bias_class1,-0.291209,0.995035,-0.918641
cat_log_min_bias_class2,-0.039526,-0.931771,0.985016



=== cat_pairwise_te_bias_from_shared_min_proba vs lgb_digit_te_min_biased : OOF classwise cross-corr ===


,lgb_digit_te_min_biased_class0,lgb_digit_te_min_biased_class1,lgb_digit_te_min_biased_class2
cat_pairwise_te_bias_from_shared_min_proba_class0,0.971902,-0.254501,-0.123702
cat_pairwise_te_bias_from_shared_min_proba_class1,-0.287913,0.996963,-0.921963
cat_pairwise_te_bias_from_shared_min_proba_class2,-0.086618,-0.920920,0.992550



=== xgb_digit_orderedte_min_biased vs lgb_digit_te_min_biased : OOF classwise cross-corr ===


,lgb_digit_te_min_biased_class0,lgb_digit_te_min_biased_class1,lgb_digit_te_min_biased_class2
xgb_digit_orderedte_min_biased_class0,0.990794,-0.272963,-0.112047
xgb_digit_orderedte_min_biased_class1,-0.290260,0.999520,-0.923685
xgb_digit_orderedte_min_biased_class2,-0.087583,-0.925831,0.998044


In [14]:
# 行和を確認する
for name, (oof, pred) in model_dict.items():
    print(name)
    print("  oof shape:", oof.shape)
    print("  pred shape:", pred.shape)
    print("  oof row sum head:", np.round(oof[:5].sum(axis=1), 6))
    print("  pred row sum head:", np.round(pred[:5].sum(axis=1), 6))
    print("  oof min/max:", float(oof.min()), float(oof.max()))
    print()

cat_strict_baseline_bias
  oof shape: (630000, 3)
  pred shape: (270000, 3)
  oof row sum head: [1. 1. 1. 1. 1.]
  pred row sum head: [1. 1. 1. 1. 1.]
  oof min/max: 2.032334123214241e-08 0.999988317489624

cat_strict_baseline_bias_wide
  oof shape: (630000, 3)
  pred shape: (270000, 3)
  oof row sum head: [1. 1. 1. 1. 1.]
  pred row sum head: [1. 1. 1. 1. 1.]
  oof min/max: 4.337254999597917e-09 0.9999910593032837

cat_digit_min_bias
  oof shape: (630000, 3)
  pred shape: (270000, 3)
  oof row sum head: [1. 1. 1. 1. 1.]
  pred row sum head: [1. 1. 1. 1. 1.]
  oof min/max: 7.509937560712387e-09 0.9999952912330627

cat_log_min_bias
  oof shape: (630000, 3)
  pred shape: (270000, 3)
  oof row sum head: [1. 1. 1. 1. 1.]
  pred row sum head: [1. 1. 1. 1. 1.]
  oof min/max: 2.8126478923695686e-09 0.9999900460243225

cat_pairwise_te_bias_from_shared_min_proba
  oof shape: (630000, 3)
  pred shape: (270000, 3)
  oof row sum head: [1. 1. 1. 1. 1.]
  pred row sum head: [1. 1. 1. 1. 1.]
  oof mi